# 04 — Test GraphSAGE and GAT

Load the weights written by `03_graph_models.ipynb` and score `dataset/merged_test.parquet`. Holdout charts are in `03_graph_models_result.ipynb`.

1. Rebuild the test parquet if it is still a copy of train.
2. Recreate the StandardScaler on the first 80% of `merged_train`.
3. Reload-check the labeled 20% holdout.
4. Build the stitched identity graph on test only (inductive) and write probabilities.

The IEEE test file has **no `isFraud` labels**.

Kernel: PyTorch (`.gnn-venv`).


In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

ROOT = Path("/content/drive/MyDrive/minor-thesis") if IS_COLAB else Path.cwd()
DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Root: {ROOT}")


## Helpers (same graph and models as notebook 03)


In [ ]:
"""Turn RF / LightGBM / XGBoost feature importance into graph edge keys.

Identity *pieces* (card, addr, email, device, uid) become:
- single-key temporal edges, and
- concatenated pair keys ("string the pieces together").

Numeric behaviour columns (C, D, amount, time) stay as *node features*.
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd

MODEL_DIR = SAVED_PATH / "ml_test_models"
SCHEMA_PATH = SAVED_PATH / "graph_edge_schema.json"

IDENTITY_PIECES = {
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "DeviceType",
    "DeviceInfo",
    "ProductCD",
    "uid",
    "uid2",
}

MISSING_VALUES = {-999, -1}
MIN_UNIQUES_SINGLE = 40
MAX_GROUP_FRAC = 0.05
TOP_PIECES = 8
TOP_PAIRS = 10
K_NEIGHBORS = 5

# uid = card1_card2_card3_card5; uid2 = uid_addr1_P_emaildomain.
# Pairing a composite with one of its own pieces just rebuilds the same chain.
UID_CONTAINED = {"card1", "card2", "card3", "card5"}
UID2_CONTAINED = UID_CONTAINED | {"uid", "addr1", "P_emaildomain"}


def _redundant_pair(a: str, b: str) -> bool:
    pair = {a, b}
    if "uid" in pair and pair & UID_CONTAINED:
        return True
    if "uid2" in pair and pair & UID2_CONTAINED:
        return True
    return False


def _latest(pattern: str) -> Path:
    hits = sorted(MODEL_DIR.glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No files matching {MODEL_DIR / pattern}")
    return hits[-1]


def load_model_importances() -> pd.DataFrame:
    import joblib

    feat_path = _latest("features_*.json")
    with open(feat_path, encoding="utf-8") as f:
        names = json.load(f)["feature_names"]

    models = {
        "RandomForest": joblib.load(_latest("randomforest_*.pkl")),
        "LightGBM": joblib.load(_latest("lightgbm_*.pkl")),
        "XGBoost": joblib.load(_latest("xgboost_*.pkl")),
    }
    rows = []
    for model_name, model in models.items():
        imp = np.asarray(model.feature_importances_, dtype=np.float64)
        if imp.sum() > 0:
            imp = imp / imp.sum()
        for feat, val in zip(names, imp):
            rows.append({"model": model_name, "feature": feat, "importance": float(val)})
    long = pd.DataFrame(rows)
    wide = long.pivot(index="feature", columns="model", values="importance").fillna(0.0)
    wide["mean"] = wide.mean(axis=1)
    wide["rank"] = wide["mean"].rank(ascending=False, method="min").astype(int)
    wide["is_identity_piece"] = wide.index.isin(IDENTITY_PIECES)
    return wide.sort_values("mean", ascending=False).reset_index()


def _nunique(series: pd.Series) -> int:
    return int(series.nunique(dropna=True))


def propose_edge_keys(df: pd.DataFrame, importance: pd.DataFrame) -> list[dict]:
    pieces = (
        importance.loc[importance["is_identity_piece"]]
        .sort_values("mean", ascending=False)["feature"]
        .tolist()
    )
    pieces = [p for p in pieces if p in df.columns][:TOP_PIECES]
    keys: list[dict] = []

    def add_key(name: str, parts: list[str], source: str, score: float) -> None:
        if any(p not in df.columns for p in parts):
            return
        if any(k["name"] == name for k in keys):
            return
        keys.append(
            {
                "name": name,
                "parts": parts,
                "source": source,
                "score": float(score),
            }
        )

    score_map = dict(zip(importance["feature"], importance["mean"]))

    for piece in pieces:
        nunq = _nunique(df[piece])
        if nunq < MIN_UNIQUES_SINGLE:
            continue
        add_key(piece, [piece], "single", score_map.get(piece, 0.0))

    pair_cands = []
    for i, a in enumerate(pieces):
        for b in pieces[i + 1 :]:
            if _redundant_pair(a, b):
                continue
            pair_cands.append((score_map.get(a, 0.0) + score_map.get(b, 0.0), a, b))
    pair_cands.sort(reverse=True)
    for score, a, b in pair_cands[:TOP_PAIRS]:
        add_key(f"link_{a}_{b}", [a, b], "pair", score)

    # Always keep the original engineered identities if present.
    for extra in ("uid", "uid2"):
        if extra in df.columns:
            add_key(extra, [extra], "engineered", score_map.get(extra, 0.0))

    keys.sort(key=lambda k: -k["score"])
    return keys


def key_series(df: pd.DataFrame, parts: list[str]) -> pd.Series:
    if len(parts) == 1:
        return df[parts[0]]
    cols = [df[p].to_numpy() for p in parts]
    return pd.Series(list(zip(*cols)), index=df.index)


def invalid_group_mask(df: pd.DataFrame, parts: list[str]) -> pd.Series:
    mask = pd.Series(False, index=df.index)
    for p in parts:
        mask |= df[p].isin(MISSING_VALUES)
    return mask


def build_temporal_knn_for_key(df: pd.DataFrame, parts: list[str], k: int = K_NEIGHBORS) -> np.ndarray:
    keys = key_series(df, parts)
    invalid = invalid_group_mask(df, parts)
    src_parts: list[np.ndarray] = []
    dst_parts: list[np.ndarray] = []
    max_group = int(len(df) * MAX_GROUP_FRAC)

    work = df.loc[~invalid, ["TransactionDT"]].copy()
    work["_k"] = keys.loc[~invalid].to_numpy()
    for _, group in work.groupby("_k", sort=False):
        n = len(group)
        if n < 2 or n > max_group:
            continue
        idx = group.sort_values("TransactionDT").index.to_numpy(dtype=np.int64, copy=False)
        for offset in range(1, min(k + 1, n)):
            left, right = idx[:-offset], idx[offset:]
            src_parts.extend((left, right))
            dst_parts.extend((right, left))
    if not src_parts:
        return np.zeros((2, 0), dtype=np.int64)
    return np.vstack([np.concatenate(src_parts), np.concatenate(dst_parts)])


def coalesce_edges(edge_index: np.ndarray, num_nodes: int) -> np.ndarray:
    if edge_index.size == 0:
        return edge_index
    packed = edge_index[0].astype(np.int64) * np.int64(num_nodes) + edge_index[1].astype(np.int64)
    _, uniq = np.unique(packed, return_index=True)
    return edge_index[:, np.sort(uniq)]


def build_union_edges(df: pd.DataFrame, edge_keys: list[dict], k: int = K_NEIGHBORS) -> tuple[np.ndarray, list[dict]]:
    stats = []
    chunks = []
    for spec in edge_keys:
        edges = build_temporal_knn_for_key(df, spec["parts"], k=k)
        stats.append(
            {
                **spec,
                "directed_edges": int(edges.shape[1]),
            }
        )
        print(
            f"  {spec['name']:30s}  {spec['source']:10s}  "
            f"parts={'+'.join(spec['parts']):40s}  edges={edges.shape[1]:,}",
            flush=True,
        )
        if edges.shape[1]:
            chunks.append(edges)
    if not chunks:
        union = np.zeros((2, 0), dtype=np.int64)
    else:
        union = coalesce_edges(np.concatenate(chunks, axis=1), len(df))
    return union, stats


def load_schema(path: Path = SCHEMA_PATH) -> dict:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def resolve_edge_keys(df: pd.DataFrame) -> list[dict]:
    """Prefer a saved schema so the GNN venv does not need LightGBM/XGBoost."""
    if SCHEMA_PATH.exists():
        schema = load_schema()
        return schema["edge_keys"]
    try:
        importance = load_model_importances()
        edge_keys = propose_edge_keys(df, importance)
        save_schema(importance, edge_keys)
        return edge_keys
    except Exception as exc:
        print(f"Could not load model importances ({exc}); falling back to uid/uid2")
        return [
            {"name": "uid", "parts": ["uid"], "source": "engineered", "score": 1.0},
            {"name": "uid2", "parts": ["uid2"], "source": "engineered", "score": 1.0},
        ]


def save_schema(importance: pd.DataFrame, edge_keys: list[dict], path: Path = SCHEMA_PATH) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "pieces": importance.loc[importance["is_identity_piece"], "feature"].head(TOP_PIECES).tolist(),
        "edge_keys": edge_keys,
        "rules": {
            "min_uniques_single": MIN_UNIQUES_SINGLE,
            "max_group_frac": MAX_GROUP_FRAC,
            "k_neighbors": K_NEIGHBORS,
            "top_pieces": TOP_PIECES,
            "top_pairs": TOP_PAIRS,
        },
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    importance.to_parquet(SAVED_PATH / "model_column_importance.parquet", index=False)
    return path




def to_csr(edge_index, num_nodes):
    if edge_index.size == 0:
        indptr = np.zeros(num_nodes + 1, dtype=np.int64)
        indices = np.zeros(0, dtype=np.int64)
        return indptr, indices
    order = np.argsort(edge_index[0], kind="mergesort")
    src = edge_index[0, order]
    dst = edge_index[1, order]
    counts = np.bincount(src, minlength=num_nodes)
    indptr = np.zeros(num_nodes + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])
    return indptr, dst.astype(np.int64)

"""Train GraphSAGE and GAT on IEEE-CIS fraud transactions.

Graph construction
------------------
Nodes are transactions. Node features match the compact "Remove id + V"
tabular set. Edges are a union of temporal k-NN graphs, one per identity key.

Keys are chosen from RF / LightGBM / XGBoost feature importance:
identity *pieces* (card, addr, email, device, uid) become single-key edges
and concatenated pair keys. C/D/amount/time stay as node features only.

Fallback if the importance schema is missing: uid and uid2 only.
"""

import gc
import json
import random
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from torch import nn

warnings.filterwarnings("ignore")

RESULTS_DIR = SAVED_PATH / "graph_models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
K_NEIGHBORS = 5
HIDDEN = 64
DROPOUT = 0.3
EPOCHS_SAGE = 25
EPOCHS_GAT = 15
PATIENCE = 6
BATCH_SIZE = 2048
NUM_NEIGHBORS = 10
LR = 1e-3
WEIGHT_DECAY = 5e-4


def set_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def select_node_features(columns: list[str]) -> list[str]:
    """Compact set aligned with 02_ml_models.ipynb (drop id_* and V*)."""
    cols = [
        c
        for c in columns
        if c not in {"isFraud", "TransactionID", "uid", "uid2"}
        and not c.startswith("id")
        and not c.startswith("V")
    ]
    # Keep identity keys as features as well; they are also used for edges.
    for extra in ("uid", "uid2"):
        if extra in columns and extra not in cols:
            cols.append(extra)
    return cols


def build_temporal_knn_edges(df: pd.DataFrame, key: str, k: int = K_NEIGHBORS) -> np.ndarray:
    """Connect each txn to up to k previous/next txns with the same identity key."""
    src_parts: list[np.ndarray] = []
    dst_parts: list[np.ndarray] = []
    for _, group in df.groupby(key, sort=False):
        n = len(group)
        if n < 2:
            continue
        idx = group.sort_values("TransactionDT").index.to_numpy(dtype=np.int64, copy=False)
        for offset in range(1, min(k + 1, n)):
            left, right = idx[:-offset], idx[offset:]
            src_parts.extend((left, right))
            dst_parts.extend((right, left))
    if not src_parts:
        return np.zeros((2, 0), dtype=np.int64)
    src = np.concatenate(src_parts)
    dst = np.concatenate(dst_parts)
    return np.vstack([src, dst])


def coalesce_edges(edge_index: np.ndarray, num_nodes: int) -> np.ndarray:
    if edge_index.size == 0:
        return edge_index
    key = edge_index[0].astype(np.int64) * np.int64(num_nodes) + edge_index[1].astype(np.int64)
    _, uniq = np.unique(key, return_index=True)
    return edge_index[:, np.sort(uniq)]


def to_csr(edge_index: np.ndarray, num_nodes: int):
    if edge_index.size == 0:
        indptr = np.zeros(num_nodes + 1, dtype=np.int64)
        indices = np.zeros(0, dtype=np.int64)
        return indptr, indices
    order = np.argsort(edge_index[0], kind="mergesort")
    src = edge_index[0, order]
    dst = edge_index[1, order]
    counts = np.bincount(src, minlength=num_nodes)
    indptr = np.zeros(num_nodes + 1, dtype=np.int64)
    np.cumsum(counts, out=indptr[1:])
    return indptr, dst.astype(np.int64)


def sample_subgraph(seeds: np.ndarray, indptr: np.ndarray, indices: np.ndarray, num_neighbors: int, num_hops: int = 2):
    """Neighbor sampling used by GraphSAGE/GAT mini-batches."""
    rng = np.random.default_rng()
    node_to_local = {int(n): i for i, n in enumerate(seeds)}
    nodes = [int(n) for n in seeds]
    hop_nodes = list(seeds)
    for _ in range(num_hops):
        nxt = []
        for n in hop_nodes:
            start, end = int(indptr[n]), int(indptr[n + 1])
            neigh = indices[start:end]
            if neigh.size == 0:
                continue
            if neigh.size > num_neighbors:
                chosen = rng.choice(neigh, size=num_neighbors, replace=False)
            else:
                chosen = neigh
            for m in chosen:
                m = int(m)
                if m not in node_to_local:
                    node_to_local[m] = len(nodes)
                    nodes.append(m)
                nxt.append(m)
        hop_nodes = nxt
    nodes_arr = np.asarray(nodes, dtype=np.int64)
    src, dst = [], []
    for n in nodes_arr:
        local_n = node_to_local[int(n)]
        start, end = int(indptr[n]), int(indptr[n + 1])
        for m in indices[start:end]:
            m = int(m)
            if m in node_to_local:
                src.append(local_n)
                dst.append(node_to_local[m])
    if not src:
        eidx = torch.zeros((2, 0), dtype=torch.long)
    else:
        eidx = torch.tensor([src, dst], dtype=torch.long)
    return nodes_arr, eidx


def scatter_sum(src: torch.Tensor, index: torch.Tensor, dim_size: int) -> torch.Tensor:
    out = torch.zeros(dim_size, *src.shape[1:], device=src.device, dtype=src.dtype)
    out.index_add_(0, index, src)
    return out


def scatter_softmax(src: torch.Tensor, index: torch.Tensor, dim_size: int) -> torch.Tensor:
    """Numerically stable softmax over incoming edges of each destination node."""
    idx = index.view(-1, *([1] * (src.dim() - 1))).expand_as(src)
    maxes = torch.full((dim_size,) + src.shape[1:], torch.finfo(src.dtype).min, device=src.device, dtype=src.dtype)
    maxes.scatter_reduce_(0, idx, src, reduce="amax", include_self=True)
    exp = (src - maxes.index_select(0, index)).exp()
    denom = scatter_sum(exp, index, dim_size).clamp_min(1e-16)
    return exp / denom.index_select(0, index)


class SAGEConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.lin_self = nn.Linear(in_channels, out_channels)
        self.lin_neigh = nn.Linear(in_channels, out_channels)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        if edge_index.numel() == 0:
            return self.lin_self(x)
        row, col = edge_index[0], edge_index[1]
        neigh = scatter_sum(x[col], row, x.size(0))
        deg = scatter_sum(torch.ones(col.size(0), 1, device=x.device, dtype=x.dtype), row, x.size(0)).clamp_min(1.0)
        neigh = neigh / deg
        return self.lin_self(x) + self.lin_neigh(neigh)


class GATConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, heads: int = 4, dropout: float = 0.1, concat: bool = True):
        super().__init__()
        self.heads = heads
        self.out_channels = out_channels
        self.concat = concat
        self.dropout = dropout
        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.att_src = nn.Parameter(torch.empty(1, heads, out_channels))
        self.att_dst = nn.Parameter(torch.empty(1, heads, out_channels))
        self.bias = nn.Parameter(torch.zeros(heads * out_channels if concat else out_channels))
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        n, h, c = x.size(0), self.heads, self.out_channels
        xh = self.lin(x).view(n, h, c)
        if edge_index.numel() == 0:
            out = xh.reshape(n, h * c) if self.concat else xh.mean(dim=1)
            return out + self.bias
        row, col = edge_index[0], edge_index[1]
        alpha = (xh[row] * self.att_src).sum(-1) + (xh[col] * self.att_dst).sum(-1)
        alpha = F.leaky_relu(alpha, 0.2)
        alpha = scatter_softmax(alpha, row, n)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        msg = xh[col] * alpha.unsqueeze(-1)
        out = scatter_sum(msg, row, n)
        if self.concat:
            out = out.reshape(n, h * c)
        else:
            out = out.mean(dim=1)
        return out + self.bias


class GraphSAGE(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int = HIDDEN, dropout: float = DROPOUT):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.dropout = dropout
        self.classifier = nn.Linear(hidden_channels, 1)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x).squeeze(-1)


class GATNet(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int = 32, heads: int = 2, dropout: float = DROPOUT):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout, concat=True)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, dropout=dropout, concat=False)
        self.dropout = dropout
        self.classifier = nn.Linear(hidden_channels, 1)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x).squeeze(-1)


def train_one_model(
    name: str,
    model: nn.Module,
    x_all: torch.Tensor,
    y_all: np.ndarray,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    edge_index: torch.Tensor,
    pos_weight: torch.Tensor,
    device: torch.device,
    epochs: int,
):
    """One full-graph forward/backward per epoch (fits 590k nodes × ~6M edges)."""
    model = model.to(device)
    x = x_all.to(device)
    eidx = edge_index.to(device)
    y_t = torch.from_numpy(y_all.astype(np.float32)).to(device)
    train_t = torch.from_numpy(train_idx.astype(np.int64)).to(device)
    val_t = torch.from_numpy(val_idx.astype(np.int64)).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
    history = []
    best_auc = -1.0
    best_state = None
    best_epoch = 0
    stalled = 0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, eidx)
        loss = criterion(logits[train_t], y_t[train_t])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x, eidx)[val_t]
            y_prob = torch.sigmoid(val_logits).cpu().numpy()
        y_true = y_all[val_idx]
        auc = roc_auc_score(y_true, y_prob)
        prauc = average_precision_score(y_true, y_prob)
        avg_loss = float(loss.item())
        history.append({"epoch": epoch, "loss": avg_loss, "roc_auc": auc, "pr_auc": prauc})
        print(f"{name} epoch {epoch:02d}/{epochs} | loss {avg_loss:.4f} | val ROC-AUC {auc:.4f} | PR-AUC {prauc:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stalled = 0
        else:
            stalled += 1
            if stalled >= PATIENCE:
                print(f"{name} early stop at epoch {epoch} (best {best_auc:.4f} @ {best_epoch})")
                break
        if device.type == "cuda":
            torch.cuda.empty_cache()

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(x, eidx)[val_t]).cpu().numpy()
    y_true = y_all[val_idx]
    y_pred = (y_prob >= 0.5).astype(np.int32)
    cm = confusion_matrix(y_true, y_pred)
    metrics = {
        "model": name,
        "best_epoch": int(best_epoch),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }
    print(f"\n{name} best metrics:")
    for k, v in metrics.items():
        if k != "model":
            print(f"  {k}: {v}")
    print(classification_report(y_true, y_pred, target_names=["Legitimate", "Fraud"], digits=4, zero_division=0))
    return model, metrics, history, y_true, y_prob


def main():
    set_seed()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    train = pd.read_parquet(DATASET_PATH / "merged_train.parquet")
    train = train.sort_values("TransactionDT").reset_index(drop=True)
    print(f"Train rows: {len(train):,}  fraud rate: {train['isFraud'].mean():.4f}")

    feature_cols = select_node_features(list(train.columns))
    print(f"Node features ({len(feature_cols)}): {feature_cols}")

    y = train["isFraud"].to_numpy(dtype=np.int64)
    split = int(len(train) * 0.8)
    train_idx = np.arange(0, split)
    val_idx = np.arange(split, len(train))
    print(f"Temporal split  train={len(train_idx):,}  valid={len(val_idx):,}")
    print(f"Fraud rate train={y[train_idx].mean():.4f}  valid={y[val_idx].mean():.4f}")

    scaler = StandardScaler()
    X = train[feature_cols].to_numpy(dtype=np.float32)
    X[train_idx] = scaler.fit_transform(X[train_idx])
    X[val_idx] = scaler.transform(X[val_idx])
    x_all = torch.from_numpy(X)

    print("Building importance-stitched temporal k-NN edges...", flush=True)
    edge_keys = resolve_edge_keys(train)
    print("Edge keys:", flush=True)
    for spec in edge_keys:
        print(f"  {spec['source']:10s}  {spec['name']}  <-  {' + '.join(spec['parts'])}", flush=True)
    edge_index, edge_stats = build_union_edges(train, edge_keys, k=K_NEIGHBORS)
    print(f"union unique directed edges: {edge_index.shape[1]:,}", flush=True)

    indptr, csr_indices = to_csr(edge_index, len(train))
    deg = np.diff(indptr)
    print(
        f"avg degree {deg.mean():.2f}  max {deg.max()}  isolated {(deg == 0).sum():,}"
    )
    edge_index_t = torch.from_numpy(edge_index.astype(np.int64))
    del edge_index, indptr, csr_indices
    gc.collect()

    n_pos = max(int(y[train_idx].sum()), 1)
    n_neg = len(train_idx) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)
    print(f"pos_weight={float(pos_weight):.2f}")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    all_metrics = []
    histories = {}

    sage = GraphSAGE(in_channels=x_all.size(1), hidden_channels=HIDDEN, dropout=DROPOUT)
    sage, sage_metrics, sage_hist, y_true, sage_prob = train_one_model(
        "GraphSAGE",
        sage,
        x_all,
        y,
        train_idx,
        val_idx,
        edge_index_t,
        pos_weight,
        device,
        EPOCHS_SAGE,
    )
    all_metrics.append(sage_metrics)
    histories["GraphSAGE"] = sage_hist
    torch.save(sage.state_dict(), RESULTS_DIR / f"graphsage_{timestamp}.pt")
    sage.cpu()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()

    del sage
    gc.collect()

    gat_prob = None
    gat = GATNet(in_channels=x_all.size(1), hidden_channels=32, heads=2, dropout=DROPOUT)
    try:
        gat, gat_metrics, gat_hist, _, gat_prob = train_one_model(
            "GAT",
            gat,
            x_all,
            y,
            train_idx,
            val_idx,
            edge_index_t,
            pos_weight,
            device,
            EPOCHS_GAT,
        )
        all_metrics.append(gat_metrics)
        histories["GAT"] = gat_hist
        torch.save(gat.state_dict(), RESULTS_DIR / f"gat_{timestamp}.pt")
    except RuntimeError as exc:
        if not _is_oom(exc):
            raise
        print(
            f"GAT OOM on {device} with {edge_index_t.shape[1]:,} edges "
            f"({exc}). Skipping GAT; GraphSAGE weights are already saved."
        )
        gat_prob = None
    finally:
        del gat
        gc.collect()

    metadata = {
        "timestamp": timestamp,
        "device": str(device),
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "num_nodes": int(len(train)),
        "num_features": int(x_all.size(1)),
        "feature_names": feature_cols,
        "k_neighbors": K_NEIGHBORS,
        "edge_keys": edge_keys,
        "edge_stats": edge_stats,
        "graph_construction": "importance-stitched identity keys",
        "hidden": HIDDEN,
        "training_mode": "full-batch",
        "num_neighbors": NUM_NEIGHBORS,
        "pos_weight": float(pos_weight),
        "temporal_split": 0.8,
        "gat_trained": gat_prob is not None,
    }
    return _write_run_artifacts(
        timestamp,
        all_metrics,
        histories,
        y_true,
        sage_prob,
        gat_prob,
        metadata,
    )

def _is_oom(exc: BaseException) -> bool:
    msg = str(exc).lower()
    return "out of memory" in msg or "not enough memory" in msg


def _write_run_artifacts(
    timestamp: str,
    all_metrics: list,
    histories: dict,
    y_true,
    sage_prob,
    gat_prob,
    metadata: dict,
) -> pd.DataFrame:
    metrics_df = pd.DataFrame(all_metrics)
    metrics_df.to_parquet(RESULTS_DIR / f"gnn_results_{timestamp}.parquet", index=False)
    metrics_df.to_csv(RESULTS_DIR / f"gnn_results_{timestamp}.csv", index=False)

    pred = {"y_true": y_true, "graphsage_prob": sage_prob}
    if gat_prob is not None:
        pred["gat_prob"] = gat_prob
    pd.DataFrame(pred).to_parquet(RESULTS_DIR / f"gnn_val_predictions_{timestamp}.parquet", index=False)

    comparison_rows = list(all_metrics)
    ml_path = SAVED_PATH / "ml_results.parquet"
    if ml_path.exists():
        tab = pd.read_parquet(ml_path)
        tab = tab[~tab["Model"].str.contains("SMOTE|Undersampling|Original", regex=True)]
        for model_name in [
            "RF - Baseline",
            "RF - Remove V",
            "RF - Remove id + V",
            "LightGBM - Baseline",
            "LightGBM - Remove V",
            "LightGBM - Remove id + V",
            "XGBoost - Baseline",
            "XGBoost - Remove V",
            "XGBoost - Remove id + V",
        ]:
            hit = tab[tab["Model"] == model_name]
            if hit.empty:
                continue
            row = hit.iloc[0]
            comparison_rows.append(
                {
                    "model": row["Model"],
                    "best_epoch": None,
                    "accuracy": float(row["Accuracy"]),
                    "precision": float(row["Precision"]),
                    "recall": float(row["Recall"]),
                    "f1": float(row["F1"]),
                    "roc_auc": float(row["ROC-AUC"]),
                    "pr_auc": float(row["PR-AUC"]),
                    "balanced_accuracy": float(row["Balanced Accuracy"]),
                    "mcc": float(row["MCC"]),
                    "tn": int(row["TN"]) if "TN" in row else None,
                    "fp": int(row["FP"]) if "FP" in row else None,
                    "fn": int(row["FN"]) if "FN" in row else None,
                    "tp": int(row["TP"]) if "TP" in row else None,
                }
            )
    comparison = pd.DataFrame(comparison_rows)
    comparison.to_parquet(RESULTS_DIR / f"gnn_vs_tabular_{timestamp}.parquet", index=False)
    comparison.to_csv(RESULTS_DIR / f"gnn_vs_tabular_{timestamp}.csv", index=False)
    print("\nComparison:")
    print(comparison[["model", "roc_auc", "pr_auc", "f1", "recall"]].to_string(index=False))

    metadata = {**metadata, "metrics": all_metrics, "history": histories}
    with open(RESULTS_DIR / f"gnn_metadata_{timestamp}.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    print(f"\nSaved artifacts under {RESULTS_DIR}")
    return metrics_df


## Rebuild competition test if needed


In [ ]:
"""Rebuild dataset/merged_test.parquet from the IEEE test CSVs.

01_clean_dataset.ipynb accidentally wrote merged_train twice, so the current
merged_test.parquet is an identical copy of train (including isFraud).
The competition test file has no labels. Encoders are aligned to the existing
merged_train.parquet so RF / LightGBM / XGBoost / GNN features match.
"""

import gc
from pathlib import Path

import numpy as np
import pandas as pd

START_DATE = "2026-01-01"
MATCHING_BINARY = ["M1", "M2", "M3", "M5", "M6", "M7", "M8", "M9"]


def add_time_and_uid(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt = pd.to_datetime(START_DATE) + pd.to_timedelta(df["TransactionDT"], unit="s")
    df["DT_month"] = dt.dt.month
    df["DT_week"] = dt.dt.isocalendar().week.astype("int32")
    df["DT_day"] = dt.dt.day
    df["DT_weekday"] = dt.dt.weekday
    df["DT_hour"] = dt.dt.hour
    df["uid"] = (
        df["card1"].astype(str)
        + "_"
        + df["card2"].astype(str)
        + "_"
        + df["card3"].astype(str)
        + "_"
        + df["card5"].astype(str)
    )
    df["uid2"] = (
        df["uid"]
        + "_"
        + df["addr1"].astype(str)
        + "_"
        + df["P_emaildomain"].astype(str)
    )
    return df


def load_raw(split: str) -> pd.DataFrame:
    trans = pd.read_csv(DATASET_PATH / f"{split}_transaction.csv")
    ident = pd.read_csv(DATASET_PATH / f"{split}_identity.csv")
    ident.columns = [c.replace("-", "_") for c in ident.columns]
    df = trans.merge(ident, on="TransactionID", how="left")
    del trans, ident
    return add_time_and_uid(df)


def compact_feature_cols(columns) -> list[str]:
    cols = [
        c
        for c in columns
        if c not in {"isFraud", "TransactionID", "uid", "uid2"}
        and not str(c).startswith("id")
        and not str(c).startswith("V")
    ]
    for extra in ("uid", "uid2"):
        if extra in columns and extra not in cols:
            cols.append(extra)
    return cols


def test_is_train_copy(train: pd.DataFrame, test: pd.DataFrame) -> bool:
    if train.shape != test.shape:
        return False
    if "TransactionID" not in test.columns:
        return False
    return set(train["TransactionID"]) == set(test["TransactionID"])


def encode_like_train(raw_train: pd.DataFrame, encoded_train: pd.DataFrame, raw_test: pd.DataFrame) -> pd.DataFrame:
    ordered = [c for c in encoded_train.columns if c != "isFraud"]
    raw_train = raw_train.set_index("TransactionID")
    encoded_train = encoded_train.set_index("TransactionID")
    raw_test = raw_test.set_index("TransactionID")
    n = len(raw_test)
    data = {}

    for col in ordered:
        if col == "TransactionID":
            data[col] = raw_test.index.to_numpy()
            continue
        if col not in raw_test.columns:
            data[col] = np.full(n, -999, dtype=np.float32)
            continue

        if col in MATCHING_BINARY:
            data[col] = raw_test[col].map({"T": 1, "F": 0}).fillna(-1).to_numpy(dtype=np.int8)
            continue

        src = raw_train[col] if col in raw_train.columns else None
        is_cat = src is not None and (
            col in {"uid", "uid2"}
            or pd.api.types.is_object_dtype(src)
            or pd.api.types.is_string_dtype(src)
            or str(src.dtype) in {"category", "string"}
        )
        if is_cat:
            keys = src.fillna("Missing").astype(str)
            mapping = (
                pd.DataFrame({"k": keys.to_numpy(), "v": encoded_train[col].to_numpy()})
                .drop_duplicates("k")
                .set_index("k")["v"]
            )
            mapped = raw_test[col].fillna("Missing").astype(str).map(mapping)
            fill = int(encoded_train[col].max()) + 1 if encoded_train[col].notna().any() else -1
            data[col] = pd.to_numeric(mapped, errors="coerce").fillna(fill).to_numpy()
            continue

        data[col] = pd.to_numeric(raw_test[col], errors="coerce").fillna(-999).to_numpy()

    return pd.DataFrame(data)[ordered]


def reduce_like_train(df: pd.DataFrame, encoded_train: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if col not in encoded_train.columns:
            continue
        dt = encoded_train[col].dtype
        arr = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float64, copy=False)
        fill = -1.0 if pd.api.types.is_integer_dtype(dt) else -999.0
        arr = np.nan_to_num(arr, nan=fill, posinf=fill, neginf=fill)
        if pd.api.types.is_integer_dtype(dt):
            info = np.iinfo(dt)
            arr = np.clip(arr, info.min, info.max)
        df[col] = arr.astype(dt, copy=False)
    return df


def ensure_merged_test(force: bool = False) -> Path:
    train_path = DATASET_PATH / "merged_train.parquet"
    test_path = DATASET_PATH / "merged_test.parquet"
    encoded_train = pd.read_parquet(train_path)

    if test_path.exists() and not force:
        test = pd.read_parquet(test_path)
        if not test_is_train_copy(encoded_train, test) and "isFraud" not in test.columns:
            print(f"merged_test.parquet already looks like the competition test: {test.shape}")
            return test_path
        if test_is_train_copy(encoded_train, test):
            print(
                "WARNING: merged_test.parquet is an identical copy of merged_train "
                "(01_clean_dataset.ipynb saved train twice). "
                "Labeled test metrics on this file mix train and holdout rows. "
                "Re-run the test-parquet cell with force=True."
            )
            return test_path
        print(f"Using existing merged_test.parquet: {test.shape}")
        return test_path

    print("Loading raw train/test CSVs...")
    raw_train = load_raw("train")
    raw_test = load_raw("test")
    print(f"raw train {raw_train.shape}  raw test {raw_test.shape}")

    encoded = encode_like_train(raw_train, encoded_train, raw_test)
    del raw_train, raw_test
    gc.collect()
    encoded = reduce_like_train(encoded, encoded_train)
    encoded.to_parquet(test_path, index=False)
    print(f"Wrote {test_path}  shape={encoded.shape}  columns={len(encoded.columns)}")
    print("Test has no isFraud labels (IEEE-CIS competition test).")
    return test_path

ensure_merged_test(force=False)


## Score holdout and unlabeled test


In [ ]:
"""Score merged_test with the saved GraphSAGE and GAT weights."""

import json
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler


warnings.filterwarnings("ignore")

DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
GRAPH_DIR = SAVED_PATH / "graph_models"
PRED_DIR = SAVED_PATH / "test_predictions"
K_NEIGHBORS = 5


def latest_pair(directory: Path):
    metas = sorted(directory.glob("gnn_metadata_*.json"))
    pts = sorted(directory.glob("graphsage_*.pt"))
    if pts:
        stamp = pts[-1].stem.replace("graphsage_", "")
        meta_path = directory / f"gnn_metadata_{stamp}.json"
        sage_path = directory / f"graphsage_{stamp}.pt"
        gat_path = directory / f"gat_{stamp}.pt"
        if not meta_path.exists():
            metas_ok = [p for p in metas if p.stem.replace("gnn_metadata_", "") <= stamp]
            meta_path = metas_ok[-1] if metas_ok else (metas[-1] if metas else meta_path)
        return stamp, meta_path, sage_path, gat_path if gat_path.exists() else None
    if not metas:
        raise FileNotFoundError(f"No gnn_metadata_*.json in {directory}")
    meta_path = metas[-1]
    stamp = meta_path.stem.replace("gnn_metadata_", "")
    sage_path = directory / f"graphsage_{stamp}.pt"
    gat_path = directory / f"gat_{stamp}.pt"
    if not sage_path.exists():
        raise FileNotFoundError(f"Missing GraphSAGE weights for timestamp {stamp}")
    return stamp, meta_path, sage_path, gat_path if gat_path.exists() else None


def metrics_dict(name: str, y_true, y_pred, y_prob) -> dict:
    cm = confusion_matrix(y_true, y_pred)
    return {
        "Model": name,
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "ROC-AUC": float(roc_auc_score(y_true, y_prob)),
        "PR-AUC": float(average_precision_score(y_true, y_prob)),
        "Balanced Accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "MCC": float(matthews_corrcoef(y_true, y_pred)),
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1]),
    }


def print_metrics(row: dict, y_true=None, y_pred=None) -> None:
    print(f"\n{'=' * 60}\n{row['Model']}\n{'=' * 60}")
    for key in [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC",
        "Balanced Accuracy",
        "MCC",
    ]:
        print(f"{key:20s}: {row[key]:.4f}")
    print(f"Confusion [{row['TN']}, {row['FP']}; {row['FN']}, {row['TP']}]")
    if y_true is not None:
        print(
            classification_report(
                y_true, y_pred, target_names=["Legitimate", "Fraud"], digits=4, zero_division=0
            )
        )


@torch.no_grad()
def predict_graph(model, x, edge_index, device) -> np.ndarray:
    model.eval()
    logits = model(x.to(device), edge_index.to(device))
    return torch.sigmoid(logits).cpu().numpy()


def main():
    set_seed()
    PRED_DIR.mkdir(parents=True, exist_ok=True)
    ensure_merged_test(force=False)

    stamp, meta_path, sage_path, gat_path = latest_pair(GRAPH_DIR)
    with open(meta_path, encoding="utf-8") as f:
        meta = json.load(f)
    feature_cols = meta["feature_names"]
    edge_keys = meta.get("edge_keys") or [
        {"name": "uid", "parts": ["uid"], "source": "engineered", "score": 1.0},
        {"name": "uid2", "parts": ["uid2"], "source": "engineered", "score": 1.0},
    ]
    print(f"Loaded GNN timestamp {stamp}")
    print(f"Features: {len(feature_cols)}")
    print(f"Edge keys: {[k['name'] for k in edge_keys]}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    train = pd.read_parquet(DATASET_PATH / "merged_train.parquet")
    test = pd.read_parquet(DATASET_PATH / "merged_test.parquet")
    train = train.sort_values("TransactionDT").reset_index(drop=True)
    test = test.sort_values("TransactionDT").reset_index(drop=True)

    split = int(len(train) * float(meta.get("temporal_split", 0.8)))
    scaler = StandardScaler()
    scaler.fit(train.iloc[:split][feature_cols].to_numpy(dtype=np.float32))

    # Reload check on the original validation slice (labeled).
    X_train = scaler.transform(train[feature_cols].to_numpy(dtype=np.float32))
    y_train = train["isFraud"].to_numpy(dtype=np.int64)
    print("Building train graph from stitched identity keys...")
    train_np, _ = build_union_edges(train, edge_keys, k=K_NEIGHBORS)
    train_edges = torch.from_numpy(train_np.astype(np.int64))
    x_train = torch.from_numpy(X_train.astype(np.float32))
    val_idx = np.arange(split, len(train))

    sage = GraphSAGE(in_channels=len(feature_cols), hidden_channels=HIDDEN, dropout=DROPOUT)
    sage.load_state_dict(torch.load(sage_path, map_location="cpu", weights_only=True))
    sage.to(device)
    gat = None
    if gat_path is not None:
        gat = GATNet(in_channels=len(feature_cols), hidden_channels=32, heads=2, dropout=DROPOUT)
        gat.load_state_dict(torch.load(gat_path, map_location="cpu", weights_only=True))
        gat.to(device)

    rows = []
    sage_val = predict_graph(sage, x_train, train_edges, device)[val_idx]
    y_va = y_train[val_idx]
    holdout = [("GraphSAGE - train holdout 20%", sage_val)]
    if gat is not None:
        holdout.append(("GAT - train holdout 20%", predict_graph(gat, x_train, train_edges, device)[val_idx]))
    for name, prob in holdout:
        pred = (prob >= 0.5).astype(np.int32)
        row = metrics_dict(name, y_va, pred, prob)
        print_metrics(row, y_va, pred)
        rows.append(row)

    del train_edges, x_train
    torch.cuda.empty_cache() if device.type == "cuda" else None

    print("\nBuilding test graph from stitched identity keys...")
    X_test = scaler.transform(test[feature_cols].to_numpy(dtype=np.float32))
    test_np, _ = build_union_edges(test, edge_keys, k=K_NEIGHBORS)
    test_edges = torch.from_numpy(test_np.astype(np.int64))
    print(f"Test nodes {len(test):,}  directed edges {test_edges.shape[1]:,}")
    x_test = torch.from_numpy(X_test.astype(np.float32))

    sage_te = predict_graph(sage, x_test, test_edges, device)
    pred_frame = pd.DataFrame(
        {
            "TransactionID": test["TransactionID"].to_numpy(),
            "GraphSAGE_prob": sage_te,
            "GraphSAGE_pred": (sage_te >= 0.5).astype(np.int32),
        }
    )
    print(f"Test predicted fraud rate GraphSAGE: {pred_frame['GraphSAGE_pred'].mean():.4f}")
    test_holdout = [("GraphSAGE - merged_test", sage_te)]
    if gat is not None:
        gat_te = predict_graph(gat, x_test, test_edges, device)
        pred_frame["GAT_prob"] = gat_te
        pred_frame["GAT_pred"] = (gat_te >= 0.5).astype(np.int32)
        print(f"Test predicted fraud rate GAT      : {pred_frame['GAT_pred'].mean():.4f}")
        test_holdout.append(("GAT - merged_test", gat_te))

    if "isFraud" in test.columns:
        y_te = test["isFraud"].to_numpy()
        for name, prob in test_holdout:
            pred = (prob >= 0.5).astype(np.int32)
            row = metrics_dict(name, y_te, pred, prob)
            print_metrics(row, y_te, pred)
            rows.append(row)
    else:
        print("merged_test has no isFraud labels — writing probabilities only.")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    metrics_df = pd.DataFrame(rows)
    metrics_df.to_parquet(PRED_DIR / f"graph_test_metrics_{timestamp}.parquet", index=False)
    metrics_df.to_csv(PRED_DIR / f"graph_test_metrics_{timestamp}.csv", index=False)
    pred_frame.to_parquet(PRED_DIR / f"graph_test_predictions_{timestamp}.parquet", index=False)
    pred_frame.to_csv(PRED_DIR / f"graph_test_predictions_{timestamp}.csv", index=False)
    print(metrics_df.to_string(index=False))
    print(f"\nSaved -> {PRED_DIR}")
    return metrics_df, pred_frame

metrics, preds = main()
display(metrics)
print(preds.head())


## Saved files

- `saved/test_predictions/graph_test_metrics_*.csv` — labeled holdout reload check
- `saved/test_predictions/graph_test_predictions_*.csv` — test `TransactionID` + probabilities
